In [1]:
import sys
sys.path.append("../scripts")  # project root so we can import data_collection

import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
from scipy.stats import poisson
from sqlalchemy import text
from sklearn.linear_model import PoissonRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from data_collection import engine

In [2]:
# ── Load base dataset ──
with engine.connect() as conn:
    df = pd.read_sql(text("""
        SELECT
            w.game_id,
            w.date,
            w.season,
            w.home_away,
            w.tb,
            w.opponent_id,
            p.throws,
            p.era,
            p.whip,
            p.k_per_9,
            pf.park_factor,
            pf.park_factor_hr
        FROM witt_game_logs w
        JOIN pitcher_game_logs p ON w.game_id = p.game_id
        JOIN park_factors pf ON (
            CASE WHEN w.home_away = 'home' THEN 118
                 ELSE w.opponent_id
            END = pf.team_id
        )
        ORDER BY w.date
    """), conn)

print(df.shape)
print(df.head())

(626, 12)
   game_id        date  season home_away  tb  opponent_id throws  era  whip  \
0   662766  2022-04-07    2022      home   2          114      R  4.2   1.3   
1   662765  2022-04-09    2022      home   0          114      R  4.2   1.3   
2   662755  2022-04-10    2022      home   2          114      R  4.2   1.3   
3   662754  2022-04-11    2022      home   0          114      R  4.2   1.3   
4   662017  2022-04-12    2022      away   0          138      R  4.2   1.3   

   k_per_9  park_factor  park_factor_hr  
0      8.8           97             212  
1      8.8           97             212  
2      8.8           97             212  
3      8.8           97             212  
4      8.8           97              56  


In [3]:
# ── Feature engineering ──
df = df.sort_values("date").reset_index(drop=True)

df["tb_avg_7"]  = df["tb"].shift(1).rolling(7,  min_periods=3).mean()
df["tb_avg_15"] = df["tb"].shift(1).rolling(15, min_periods=7).mean()
df["tb_lag1"]   = df["tb"].shift(1)
df["is_home"]   = (df["home_away"] == "home").astype(int)
df["pitcher_r"] = (df["throws"] == "R").astype(int)

df_model = df.dropna(subset=["tb_avg_7", "tb_avg_15", "tb_lag1"]).reset_index(drop=True)

print(df_model.shape)

(619, 17)


In [4]:
# ── Define features and target ──
# Fewer features this time — park factor breakdown columns dropped,
# keeping only the most meaningful ones for a small dataset
FEATURES = [
    "tb_lag1",
    "tb_avg_7",
    "tb_avg_15",
    "is_home",
    "pitcher_r",
    "era",
    "whip",
    "k_per_9",
    "park_factor",
    "park_factor_hr",
]

TARGET = "tb"

X = df_model[FEATURES]
y = df_model[TARGET]

print(X.shape)
print(y.describe())

(619, 10)
count    619.000000
mean       2.017771
std        1.993032
min        0.000000
25%        0.500000
50%        2.000000
75%        3.000000
max       11.000000
Name: tb, dtype: float64


In [5]:
# ── Scale features ──
# Poisson regression is a linear model under the hood — unlike XGBoost,
# it is sensitive to feature scale so we need to standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=FEATURES)

In [6]:
# ── Baseline: naive mean predictor ──
naive_pred = np.full(len(y), y.mean())
naive_mae  = mean_absolute_error(y, naive_pred)
print(f"Naive baseline MAE (always predict mean): {naive_mae:.3f}")

Naive baseline MAE (always predict mean): 1.511


In [7]:
# ── Cross-validation with TimeSeriesSplit ──
model = PoissonRegressor(alpha=1.0, max_iter=300)

tscv = TimeSeriesSplit(n_splits=5)

cv_mae  = -cross_val_score(model, X_scaled, y, cv=tscv, scoring="neg_mean_absolute_error")
cv_rmse = -cross_val_score(model, X_scaled, y, cv=tscv, scoring="neg_root_mean_squared_error")

print(f"CV MAE  per fold: {[round(v,3) for v in cv_mae]}")
print(f"CV RMSE per fold: {[round(v,3) for v in cv_rmse]}")
print(f"\nMAE  mean ± std: {cv_mae.mean():.3f} ± {cv_mae.std():.3f}")
print(f"RMSE mean ± std: {cv_rmse.mean():.3f} ± {cv_rmse.std():.3f}")
print(f"\nBaseline MAE:    {naive_mae:.3f}")
print(f"Improvement:     {naive_mae - cv_mae.mean():.3f}")

CV MAE  per fold: [np.float64(1.387), np.float64(1.739), np.float64(1.621), np.float64(1.351), np.float64(1.412)]
CV RMSE per fold: [np.float64(1.815), np.float64(2.308), np.float64(2.27), np.float64(1.784), np.float64(1.765)]

MAE  mean ± std: 1.502 ± 0.152
RMSE mean ± std: 1.989 ± 0.246

Baseline MAE:    1.511
Improvement:     0.009


In [8]:
# ── Fit on full dataset ──
model.fit(X_scaled, y)

# Poisson model predicts lambda (expected TB rate)
lambda_preds  = model.predict(X_scaled)
insample_mae  = mean_absolute_error(y, lambda_preds)
insample_rmse = root_mean_squared_error(y, lambda_preds)

print(f"In-sample MAE:  {insample_mae:.3f}")
print(f"In-sample RMSE: {insample_rmse:.3f}")
print(f"CV MAE:         {cv_mae.mean():.3f}")
print(f"\nGap (CV - in-sample): {cv_mae.mean() - insample_mae:.3f}")
print("(large gap = overfitting)")

In-sample MAE:  1.507
In-sample RMSE: 1.962
CV MAE:         1.502

Gap (CV - in-sample): -0.005
(large gap = overfitting)


In [9]:
# ── Derive over/under probabilities from lambda ──
# This is the key advantage of Poisson — from one model we get
# the probability of any threshold analytically
df_model["lambda"]    = lambda_preds
df_model["p_over_0"]  = 1 - poisson.cdf(0, lambda_preds)   # P(TB >= 1)
df_model["p_over_1"]  = 1 - poisson.cdf(1, lambda_preds)   # P(TB >= 2, i.e. over 1.5)
df_model["p_over_2"]  = 1 - poisson.cdf(2, lambda_preds)   # P(TB >= 3, i.e. over 2.5)

print(df_model[["date", "tb", "lambda", "p_over_0", "p_over_1", "p_over_2"]].head(10))

print(f"\nMean predicted lambda: {lambda_preds.mean():.3f}")
print(f"Mean P(over 1.5 TB):   {df_model['p_over_1'].mean():.3f}")
print(f"Actual rate over 1.5:  {(y >= 2).mean():.3f}")

         date  tb    lambda  p_over_0  p_over_1  p_over_2
0  2022-04-16   2  2.381101  0.907551  0.687422  0.425346
1  2022-04-19   0  2.045698  0.870710  0.606222  0.335691
2  2022-04-20   0  2.299718  0.899713  0.669081  0.403886
3  2022-04-21   2  1.995945  0.864115  0.592896  0.322226
4  2022-04-22   1  1.838795  0.840991  0.548606  0.279788
5  2022-04-23   2  1.622971  0.802688  0.482457  0.222595
6  2022-04-24   1  1.618260  0.801757  0.480948  0.221371
7  2022-04-26   3  1.960171  0.859166  0.583106  0.312544
8  2022-04-27   1  1.640569  0.806130  0.488074  0.227177
9  2022-04-28   1  1.683990  0.814368  0.501766  0.238557

Mean predicted lambda: 2.018
Mean P(over 1.5 TB):   0.595
Actual rate over 1.5:  0.507


In [10]:
# ── Feature coefficients ──
# In Poisson regression coefficients are on log scale —
# exp(coef) tells you the multiplier effect on expected TB
coefs = pd.Series(model.coef_, index=FEATURES).sort_values()
print("Coefficients (log scale):")
print(coefs)
print("\nMultiplier effect on lambda (exp of coef):")
print(np.exp(coefs).round(3))

Coefficients (log scale):
k_per_9          -0.063262
tb_lag1          -0.041653
tb_avg_7         -0.024245
whip              0.005165
era               0.022410
park_factor_hr    0.028497
park_factor       0.033669
is_home           0.037119
pitcher_r         0.041840
tb_avg_15         0.047530
dtype: float64

Multiplier effect on lambda (exp of coef):
k_per_9           0.939
tb_lag1           0.959
tb_avg_7          0.976
whip              1.005
era               1.023
park_factor_hr    1.029
park_factor       1.034
is_home           1.038
pitcher_r         1.043
tb_avg_15         1.049
dtype: float64


In [11]:
# ── Calibration check ──
# Does the model's predicted probability match the actual hit rate?
# Bin predictions into deciles and compare
df_model["pred_bin"] = pd.qcut(df_model["p_over_1"], q=5, labels=False)
calibration = df_model.groupby("pred_bin").agg(
    mean_predicted=("p_over_1", "mean"),
    actual_rate=("tb", lambda x: (x >= 2).mean()),
    n=("tb", "count")
).round(3)

print(calibration)
print("\n(well-calibrated = predicted and actual rates close in each bin)")

          mean_predicted  actual_rate    n
pred_bin                                  
0                  0.505        0.395  124
1                  0.563        0.516  124
2                  0.598        0.545  123
3                  0.632        0.460  124
4                  0.677        0.621  124

(well-calibrated = predicted and actual rates close in each bin)


In [12]:
# ── Save model and scaler ──
# Save both — scaler is needed to transform new game inputs before predicting
joblib.dump(model,  "../models/witt_poisson_model.pkl")
joblib.dump(scaler, "../models/witt_poisson_scaler.pkl")
print("Model saved to models/witt_poisson_model.pkl")
print("Scaler saved to models/witt_poisson_scaler.pkl")

Model saved to models/witt_poisson_model.pkl
Scaler saved to models/witt_poisson_scaler.pkl
